#### Control: Provides deterministic decision-making and process flow control. This component handles if/then logic, routing based on conditions, and process orchestration for predictable behavior.

#### NOTE: i am using github openai for this example.

In [8]:
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from pydantic import BaseModel
from typing import Literal
import json

In [3]:
TOKEN = os.getenv('GITHUB_TOKEN')
ENDPOINT = os.getenv('GITHUB_ENDPOINT')
MODEL = os.getenv('GITHUB_MODEL_NAME')

client = OpenAI(
    base_url=ENDPOINT,
    api_key=TOKEN,
)

In [13]:
class IntentClassification(BaseModel):
    intent: Literal["question", "request", "complaint"]
    confidence: float
    reasoning: str
    result: str = None

def route_based_on_intent(user_input: str) -> IntentClassification:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "Classify user input into one of three categories: question, request, or complaint. Provide your reasoning and confidence level. Reply ONLY with a valid JSON object: {\"intent\": str, \"confidence\": float, \"reasoning\": str}",
            },
            {"role": "user", "content": user_input}
        ],
        response_format={"type": "json_object"}
    )

    classification = IntentClassification.model_validate(json.loads(response.choices[0].message.content))
    intent = classification.intent

    if intent == "question":
        result = answer_question(user_input)
    elif intent == "request":
        result = process_request(user_input)
    elif intent == "complaint":
        result = handle_complaint(user_input)
    else:
        result = "I'm not sure how to handle that input."

    return IntentClassification(
        intent=intent,
        confidence=classification.confidence,
        reasoning=classification.reasoning,
        result=result
    )

def answer_question(question: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer the user's question. Keep the response short."},
            {"role": "user", "content": question}
        ]
    )

    return response.choices[0].message.content

def process_request(request: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Process the user's request. Keep the response short."},
            {"role": "user", "content": request}
        ]
    )

    return response.choices[0].message.content

def handle_complaint(complaint: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Address the user's complaint. Keep the response short."},
            {"role": "user", "content": complaint}
        ]
    )

    return response.choices[0].message.content

In [14]:
test_inputs = [
    "What is machine learning?",
    "Please schedule a meeting for tomorrow",
    "I'm unhappy with the service quality",
]

for user_input in test_inputs:
    result = route_based_on_intent(user_input)
    print("=" * 20)
    print(f"Input: {user_input}")
    print(f"Intent: {result.intent}, Confidence: {result.confidence}, Reasoning: {result.reasoning}")
    print(f"Result: {result.result}\n")
    print("=" * 20)

Input: What is machine learning?
Intent: question, Confidence: 1.0, Reasoning: The input is asking for information about what machine learning is, which is an inquiry and thus a question.
Result: Machine learning is a type of artificial intelligence that enables computers to learn from data and improve their performance at tasks without being explicitly programmed. It involves creating algorithms that can identify patterns and make predictions based on input data.

Input: Please schedule a meeting for tomorrow
Intent: request, Confidence: 0.98, Reasoning: The user is asking for an action to be performed, specifically to schedule a meeting, which fits the definition of a request.
Result: Sure! What time would you like to schedule the meeting, and who should be invited?

Input: I'm unhappy with the service quality
Intent: complaint, Confidence: 0.99, Reasoning: The user expresses dissatisfaction with the service, which is characteristic of a complaint rather than a question or a request.